# Benchmark tables and event-stream comparisons
Each dataset has a main table and a separate spatial voxel comparison. Chamfer Full is the only Chamfer variant shown. Edit TABLE_COLUMNS in the table cell and rerun that cell. The plots below compare one sequence using the benchmark crop and physical frame timestamps.

Each number is a sequence-weighted mean. Within a column, methods use the same sequences. The voxel table also matches sequences across available scales for each metric and polarity. Headers show the number of shared sequences. Only finite, completed scores are included. Fano scores with partial spatial support are included and flagged in the report inventory.

Blue marks the four best rank levels (darkest first, equal scores share a rank). For descriptive columns (including Same-ts), cells show the mean raw value and colors rank the mean absolute difference from ground truth across matched sequences. Ground truth is a reference row. Count ratio shows the mean ratio and, in parentheses, the mean per-sequence absolute deviation from 1 used for ranking. A dash indicates unavailable data. See the report inventory to check coverage and the selected runs.

In [ ]:
from pathlib import Path
import json
from html import escape

import numpy as np
import pandas as pd
from IPython.display import HTML, display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "benchmarks/run_metric.py").is_file()), Path.cwd())
REPORT_ROOT = root / "reports"
SPLIT = "test"
# Optional: choose a specific run directory name per (dataset, metric, block size).
# Use None for the block size of a non-voxel metric.
RUN_OVERRIDE = {}
DATASET = "bsergb"
SEQUENCE = None       # None selects the first sequence with a saved event stream.
METHODS = None        # None discovers all saved methods. Example: ["v2ce", "evtween_v5.2_student"]
START_FRAME = 0       # Zero-based index of the first frame in the detail view.
N_FRAMES = 17         # 17 frames span 16 frame intervals.
RATE_BIN_S = 0.001    # Event-rate bin width in seconds, including the detail view.
DATA_ROOT = None      # Optional path containing dataset/split/sequence folders.
PRED_ROOT = root / "data_test_evt"


In [ ]:
MAIN = {
    "chamfer_full": "Chamfer Full ↓", "iei_distance": "IEI distance ↓",
    "fano_factor": "Fano error ↓", "event_count_ratio": "Count ratio ≈1",
    "event_count": "Event count", "event_rate": "Event rate (events/s)",
    "on_percentage": "ON (%)", "isi_median_ms": "Median ISI (ms)",
    "isi_cv": "ISI CV", "same_ts": "Same-ts fraction",
}
VOXEL = {"voxel_mae": "MAE", "voxel_rmse": "RMSE"}
CHANNELS = {"value": "Signed", "value_on": "ON", "value_off": "OFF"}
LOWER = {"chamfer_full", "iei_distance", "fano_factor", *VOXEL}
runs = {}
for cfg_path in sorted(REPORT_ROOT.glob("*/*/*/config.json")):
    cfg = json.loads(cfg_path.read_text())
    metric = cfg["metric"]
    if cfg["split"] != SPLIT or metric not in {*MAIN, *VOXEL}:
        continue
    size = cfg.get("options", {}).get("block_size", 1) if metric in VOXEL else None
    csv = cfg_path.with_name("results.csv")
    if csv.is_file():
        runs.setdefault((cfg["dataset"], metric, size), []).append(csv)

reports, inventory = {}, []
for key, paths in runs.items():
    override = RUN_OVERRIDE.get(key)
    choices = [p for p in paths if override is None or p.parent.name == override]
    if not choices:
        raise ValueError(f"Unknown run {override!r} for {key}")
    path = max(choices, key=lambda p: (p.stat().st_mtime_ns, str(p)))
    raw = pd.read_csv(path)
    if raw.duplicated(["sequence", "method"]).any():
        raise ValueError(f"Duplicate sequence/method rows in {path}")
    ok = raw.status.eq("ok") | ((key[1] == "fano_factor") & raw.status.eq("partial_support"))
    rows = raw.copy()
    for field in CHANNELS:
        rows[field] = pd.to_numeric(rows.get(field, np.nan), errors="coerce")
        rows.loc[~ok | ~np.isfinite(rows[field]), field] = np.nan
    reports[key] = rows
    inventory.append(dict(dataset=key[0], metric=key[1], block=key[2], run=path.parent.name,
                          candidates=len(paths), scored=int(rows.value.notna().sum()), rows=len(raw),
                          partial_support=int(raw.status.eq("partial_support").sum())))
if not reports:
    print(f"No matching {SPLIT} reports under {REPORT_ROOT}")
if set(RUN_OVERRIDE) - set(reports):
    raise ValueError(f"Missing report overrides: {set(RUN_OVERRIDE) - set(reports)}")
datasets = sorted({key[0] for key in reports})
voxel_sizes = sorted({size for _, metric, size in reports if metric in VOXEL})
print(f"{SPLIT} | {', '.join(datasets)} | {len(reports)} selected reports")


In [ ]:
def summarize(dataset, metric, field, sizes):
    matrices, common = {}, None
    for size in sizes:
        rows = reports.get((dataset, metric, size))
        if rows is None:
            continue
        matrix = rows.pivot(index="sequence", columns="method", values=field)
        matrices[size] = matrix
        available = matrix.drop(columns="ground_truth", errors="ignore").dropna(axis=1, how="all")
        if available.shape[1]:
            complete = available.dropna().index
            common = complete if common is None else common.intersection(complete)
    columns = {}
    for size, matrix in matrices.items():
        paired = matrix.reindex([] if common is None else common)
        descriptor = metric in MAIN and metric not in LOWER and metric != "event_count_ratio"
        if descriptor:
            paired = paired.loc[paired["ground_truth"].notna()] if "ground_truth" in paired else paired.iloc[:0]
        means = paired.mean()
        ranks = pd.Series(dtype=float)
        deviation = None
        if metric == "event_count_ratio":
            deviation = paired.sub(1).abs().mean()
            ranks = deviation
        elif metric in LOWER:
            ranks = means
        elif descriptor and "ground_truth" in paired:
            ranks = paired.sub(paired["ground_truth"], axis="index").abs().mean()
        ranks = ranks.drop(labels="ground_truth", errors="ignore").rank(method="dense")
        columns[size] = (means, ranks, deviation, len(paired))
    return columns

# Precompute the summaries used by the tables.
tables = {}
for dataset in datasets:
    main, voxel = {}, {}
    for metric, label in MAIN.items():
        main[label] = summarize(dataset, metric, "value", [None]).get(None)
    for metric, short in VOXEL.items():
        for field, channel in CHANNELS.items():
            by_size = summarize(dataset, metric, field, voxel_sizes)
            for size in voxel_sizes:
                voxel[f"{short} {channel} {size}×{size}"] = by_size.get(size)
    tables[dataset] = main, voxel

In [ ]:
from fnmatch import fnmatchcase
# EDIT THIS LINE: use the printed labels or wildcards (e.g. "MAE ON *" or "RMSE Signed 4×4").
TABLE_COLUMNS = [*MAIN.values(), "MAE Signed *", "RMSE Signed *"]

def render(title, columns, selected, methods):
    labels = [label for label in columns if label in selected]
    if not labels:
        return
    palette = ["#164b80", "#75a9d6", "#b4d4ed", "#e0eef8"]
    html = [f"<h4>{escape(title)}</h4><div style='overflow-x:auto'><table><thead><tr><th>Method</th>"]
    for label in labels:
        col = columns[label]
        count = col[3] if col else 0
        html.append(f"<th>{escape(label)}<br><small>n={count}</small></th>")
    html.append("</tr></thead><tbody>")
    for method in methods:
        if method == "ground_truth" and not any(columns[l] and pd.notna(columns[l][0].get(method)) for l in labels):
            continue
        html.append(f"<tr><th>{escape(str(method))}</th>")
        for label in labels:
            col = columns[label]
            value = col[0].get(method, np.nan) if col else np.nan
            rank = col[1].get(method, np.nan) if col else np.nan
            style, number = "", "—"
            if pd.notna(value):
                number = f"{value:,.5g}"
                if col[2] is not None:
                    number += f" ({col[2][method]:.4g})"
                if pd.notna(rank) and rank <= 4:
                    color = "white" if rank == 1 else "#102b43"
                    style = f"background:{palette[int(rank)-1]};color:{color}"
            html.append(f"<td style='{style}'>{number}</td>")
        html.append("</tr>")
    display(HTML("".join(html) + "</tbody></table></div>"))

if tables:
    print("Available columns:", ", ".join(label for group in next(iter(tables.values())) for label in group))
for dataset, (main, voxel) in tables.items():
    methods = sorted({m for key, rows in reports.items() if key[0] == dataset for m in rows.method.unique()})
    for title, cols in [("Main results", main), ("Voxel errors by spatial scale", voxel)]:
        selected = [label for label in cols if any(fnmatchcase(label, pattern) for pattern in TABLE_COLUMNS)]
        if any(cols[label] is not None for label in selected):
            render(f"{dataset} · {title}", cols, selected, methods)

In [ ]:
import sys
from types import SimpleNamespace
from itertools import islice
sys.path.insert(0, str(root))
from benchmarks import run_metric as runner

pred = PRED_ROOT / DATASET / SPLIT
if not pred.is_dir():
    raise FileNotFoundError(pred)
folders = sorted(p for p in pred.iterdir() if p.is_dir())
selected = [pred / name for name in METHODS] if METHODS is not None else folders
names = sorted({f.parent.name for folder in selected for f in folder.glob("*/events.npz")})
if SEQUENCE is None and not names:
    raise ValueError("Set SEQUENCE or choose a method with saved events.")
name = SEQUENCE or names[0]
data = Path(DATA_ROOT) if DATA_ROOT else runner.v2.old.setting("EVTWEEN_DATA", runner.v2.old.LAB / "evtween/data")
args = SimpleNamespace(dataset=DATASET, split=SPLIT, data_root=DATA_ROOT)
seq = runner.Sequence(runner.source_for(name, data / DATASET / SPLIT, folders, args))
frames, meta = seq.geometry
if not (0 <= START_FRAME < len(frames)-1 and N_FRAMES >= 2 and np.isfinite(RATE_BIN_S) and RATE_BIN_S > 0):
    raise ValueError("Check START_FRAME, N_FRAMES (at least 2) and RATE_BIN_S.")
last = min(START_FRAME + N_FRAMES - 1, len(frames)-1)
lo, hi = frames[[START_FRAME, last]]
print(f"{DATASET}/{SPLIT}/{name} | frames {START_FRAME}–{last} ({last-START_FRAME+1} frames) | [{lo:.6f}, {hi:.6f}) s")

def bin_edges(a, b):
    edges = np.arange(a, b, RATE_BIN_S)
    return np.append(edges[edges < b], b)

windows = {"Full sequence": bin_edges(frames[0], frames[-1]), "Selected frames": bin_edges(lo, hi)}
rates = {key: {} for key in windows}
polarity_rates = {key: {} for key in windows}
polarity_counts = {key: {} for key in windows}
maps = {}
shape = meta["height"], meta["width"]

def collect(method, stream):
    t = stream["t"]
    on_times = t[stream["p"] > 0]
    for key, edges in windows.items():
        counts = np.diff(np.searchsorted(t, edges, side="left"))
        on = np.diff(np.searchsorted(on_times, edges, side="left"))
        off = counts - on
        widths = np.diff(edges)
        rates[key][method] = counts / widths
        polarity_rates[key][method] = (on / widths, off / widths)
        polarity_counts[key][method] = (int(on.sum()), int(off.sum()))
    a, b = np.searchsorted(t, [lo, hi])
    xy = stream["y"][a:b].astype(np.int64)*shape[1] + stream["x"][a:b]
    maps[method] = np.bincount(xy, weights=np.where(stream["p"][a:b] > 0, 1, -1),
                              minlength=shape[0]*shape[1]).reshape(shape)

collect("Ground truth", seq.gt)
for folder in selected:
    native = folder.name == "v2v"
    path = folder / name / ("voxels.npz" if native else "events.npz")
    if not path.is_file():
        print(f"Missing {folder.name}: {path}")
        continue
    runner.verify_metadata(path.parent / "input.json", meta, args, name)
    if native:
        with runner.native_planes(path, frames, *shape) as planes:
            maps[folder.name] = sum(islice(planes, START_FRAME, last), np.zeros(shape))
        print("v2v: native signed voxels are shown in the map; individual timestamps are unavailable.")
    else:
        collect(folder.name, runner.read_events(path, meta))

In [ ]:
# Signed count maps on identical coordinates and one shared color scale.
from PIL import Image
from io import BytesIO
import base64

peaks = [np.percentile(np.abs(m[m != 0]), 99) for m in maps.values() if np.any(m)]
limit = max([1.0, *peaks])
panels = []
for method, counts in maps.items():
    scaled = np.clip(counts / limit, -1, 1)
    rgb = np.stack([1 + np.minimum(scaled, 0), 1 - np.abs(scaled), 1 - np.maximum(scaled, 0)], axis=-1)
    buffer = BytesIO()
    Image.fromarray((255*rgb).astype("uint8")).save(buffer, format="PNG")
    src = base64.b64encode(buffer.getvalue()).decode()
    panels.append(f'<figure style="margin:5px"><figcaption>{escape(method)}</figcaption>'
                  f'<img style="width:100%" src="data:image/png;base64,{src}"></figure>')
print(f"Frames {START_FRAME}–{last} | ON minus OFF per pixel | red: positive, blue: negative, white: zero | saturation ±{limit:.1f}")
display(HTML('<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(235px,1fr))">' + "".join(panels) + '</div>'))

In [ ]:
# Direct notebook plots. Change to "Full sequence" to compare the entire recording.
PLOT_WINDOW = "Selected frames"
import matplotlib.pyplot as plt

edges = windows[PLOT_WINDOW]
series = rates[PLOT_WINDOW]
palette = plt.get_cmap("tab20").colors
colors = {method: ("black" if method == "Ground truth" else palette[(2 * (i - 1) % 20 + (i - 1) // 10) % 20])
          for i, method in enumerate(series)}
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True, constrained_layout=True,
                         facecolor="white")
for panel, (ax, label) in enumerate(zip(axes, ("Total", "ON", "OFF"))):
    ax.set_facecolor("white")
    for method in series:
        values = series[method] if panel == 0 else polarity_rates[PLOT_WINDOW][method][panel - 1]
        gt = method == "Ground truth"
        ax.step(edges, np.append(values, values[-1]), where="post", label=method,
                color=colors[method], linewidth=2 if gt else 1.1, zorder=5 if gt else 2)
    ax.set_ylabel(f"{label} events/s")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.2)
# A shared polarity scale makes ON and OFF amplitudes directly comparable.
top = max(ax.get_ylim()[1] for ax in axes[1:])
for ax in axes[1:]:
    ax.set_ylim(0, top)
axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
axes[-1].set(xlabel="Time since evaluation start (s)", xlim=(edges[0], edges[-1]))
fig.suptitle(f"{DATASET}/{SPLIT}/{name} · {PLOT_WINDOW} · {RATE_BIN_S * 1000:g} ms bins")
plt.show()

counts = pd.DataFrame.from_dict(polarity_counts[PLOT_WINDOW], orient="index",
                                columns=["ON events", "OFF events"])
counts.index.name = "Method"
display(counts.assign(**{"Total events": counts.sum(axis=1)}))

### Event rate over time
Select the full sequence or the frame-aligned detail view. Click a method in the legend to toggle it, drag horizontally to zoom and double-click to reset. Hover for the time bin and its event rate. The exported HTML opens offline.

In [ ]:
payload = {key: {"edges": edges.tolist(), "rates": {m: a.tolist() for m, a in rates[key].items()}}
           for key, edges in windows.items()}
title = f"{DATASET}/{SPLIT}/{name} · frames {START_FRAME}–{last} · {RATE_BIN_S*1000:g} ms bins"
document = """<!doctype html><meta charset="utf-8"><title>Event rate comparison</title>
<style>*{box-sizing:border-box}body{font:14px system-ui;color:#192a3a;background:white;margin:16px}
button,select{font:inherit;margin:3px;padding:4px 8px;background:white;border:1px solid #aab4bb;border-radius:4px}
button[aria-pressed=false]{opacity:.42}canvas{width:100%;height:350px;touch-action:none}output{display:block;min-height:35px}</style>
<h3>Event rate over time</h3><p id="info"></p><select id="scope"><option>Selected frames</option><option>Full sequence</option></select>
<button id="reset">Reset zoom</button><div id="legend"></div><canvas id="chart"></canvas><output id="readout"></output>
<script>
const data=__DATA__, title=__TITLE__;
const el=id=>document.getElementById(id), canvas=el('chart'), ctx=canvas.getContext('2d');
el('info').textContent=title;
const names=Object.keys(data['Selected frames'].rates), active=new Set(names);
const color=i=>i===0?'#151515':'hsl('+((i*137)%360)+' 68% 40%)';
let left,right,start=null,L=76,R,B=302,T=18;
const current=()=>data[el('scope').value];
function restore(){const e=current().edges; left=e[0];right=e[e.length-1];el('readout').textContent='';draw()}
function bin(e,t){let a=0,b=e.length-1;while(a+1<b){let m=(a+b)>>1;if(e[m]<=t)a=m;else b=m}return a}
function draw(){let w=canvas.clientWidth,h=350,k=devicePixelRatio||1;canvas.width=w*k;canvas.height=h*k;ctx.scale(k,k);R=w-20;
const {edges:e,rates}=current(),a=bin(e,left),b=bin(e,right);let ymax=1;
for(const n of active)for(let i=a;i<=b;i++)ymax=Math.max(ymax,rates[n][i]);
const X=t=>L+(t-left)/(right-left)*(R-L),Y=v=>B-v/ymax*(B-T);
ctx.font='11px system-ui';ctx.fillStyle='#334';ctx.strokeStyle='#ddd';
for(let j=0;j<=4;j++){let y=Y(ymax*j/4);ctx.beginPath();ctx.moveTo(L,y);ctx.lineTo(R,y);ctx.stroke();ctx.fillText((ymax*j/4).toExponential(1),3,y+4)}
for(let j=0;j<=5;j++)ctx.fillText((left+(right-left)*j/5).toFixed(4),L+(R-L)*j/5-18,B+17);
ctx.fillText('Events/s',4,12);ctx.fillText('Time since evaluation start (s)',Math.max(L,w/2-73),B+39);
ctx.save();ctx.beginPath();ctx.rect(L,T,R-L,B-T);ctx.clip();
names.forEach((n,j)=>{if(!active.has(n))return;ctx.strokeStyle=color(j);ctx.lineWidth=j===0?2.4:1.4;ctx.beginPath();
for(let i=a;i<=b;i++){let y=Y(rates[n][i]);if(i===a)ctx.moveTo(X(e[i]),y);else ctx.lineTo(X(e[i]),y);ctx.lineTo(X(e[i+1]),y)}ctx.stroke()});ctx.restore()}
names.forEach((n,j)=>{const btn=document.createElement('button');btn.textContent=n;btn.style.borderLeft='5px solid '+color(j);btn.setAttribute('aria-pressed','true');
btn.onclick=()=>{if(active.has(n))active.delete(n);else active.add(n);btn.setAttribute('aria-pressed',String(active.has(n)));draw()};el('legend').append(btn)});
function time(ev){let x=ev.clientX-canvas.getBoundingClientRect().left;return left+Math.max(0,Math.min(1,(x-L)/(R-L)))*(right-left)}
canvas.onpointerdown=ev=>{start=time(ev);canvas.setPointerCapture(ev.pointerId)};
canvas.onpointerup=ev=>{if(start===null)return;const end=time(ev);if(Math.abs(end-start)>(right-left)*.01){left=Math.min(start,end);right=Math.max(start,end);draw()}start=null};
canvas.onpointermove=ev=>{const {edges:e,rates}=current(),i=bin(e,time(ev));el('readout').textContent='['+e[i].toFixed(5)+', '+e[i+1].toFixed(5)+') s   '+[...active].map(n=>n+': '+Math.round(rates[n][i]).toLocaleString()+' events/s').join(' | ')};
canvas.ondblclick=restore;el('reset').onclick=restore;el('scope').onchange=restore;window.onresize=draw;restore();
</script>"""
document = document.replace("__TITLE__", json.dumps(title).replace("<", "\\u003c")).replace(
    "__DATA__", json.dumps(payload, separators=(",", ":"), allow_nan=False).replace("<", "\\u003c"))
output = REPORT_ROOT / "visualizations" / DATASET / SPLIT / name / f"rate_frames_{START_FRAME}_{last}.html"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(document, encoding="utf-8")
# Embed the same self-contained HTML even when Jupyter serves only the notebook directory.
display(HTML('<iframe title="Event rate comparison" style="width:100%;height:620px;border:0" srcdoc="' +
             escape(document, quote=True) + '"></iframe>'))
print("Standalone HTML:", output)
